# Capstone Phase 5 上机：商业模式与价值评估

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 核心任务

为 AI 营销 Agent SaaS「MarketingAgent Pro」构建完整投资评估，整合 Phase 4 因果效果：
1. 商业模式画布（9宫格结构化，价值主张整合Phase 4 ATE）
2. ATE -> ARPU -> DCF 估值（NPV / IRR / 回收期 / PI）
3. 蒙特卡洛模拟（传播ATE置信区间不确定性，估值分布 + 概率分析）
4. 敏感性分析（龙卷风图，含ATE和推理成本）
5. 天道推演多路径场景分析（Bull/Base/Bear，ATE CI上下界为边界）

**真实库**：numpy-financial（NPV/IRR）｜ scipy.stats（蒙特卡洛）｜ pandas + matplotlib
**真实数据**：HubSpot 2023 财报 + Jasper AI Crunchbase + OpenAI API定价 + Phase 4因果效果（ATE）


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 需要 numpy-financial, scipy, pandas, matplotlib。通常已随 conda/venv 安装。
> numpy-financial 提供 NPV/IRR 等标准金融函数。


In [ ]:
# !pip install numpy-financial scipy pandas matplotlib -q
import numpy as np
import pandas as pd
import numpy_financial as npf
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("环境就绪: numpy-financial + scipy.stats + pandas + matplotlib")


## 1. 商业模式画布（Business Model Canvas）

AI 商业模式画布在传统九宫格基础上适配 AI 原生特征：
- **收入流**：新增 outcome-based pricing（alpha=3.33%价值捕获率）
- **核心资源**：新增数据资产 + AI 模型 + 算力 + 因果实验数据
- **核心活动**：新增模型训练/评估 + Agent运维 + 因果实验
- **成本结构**：新增推理成本（持续运营成本，30%）
- **核心伙伴**：新增MCP协议连接 + A2A协作

**案例**：MarketingAgent Pro - AI 原生营销 Agent 平台
- Phase 4整合：ATE=+3.8pp转化率提升（95% CI: [2.2pp, 5.4pp]）作为价值主张核心
- 数据校准：HubSpot 2023 财报（gross margin ~78%）、Jasper AI（$125M ARR）


In [ ]:
# TODO 1：商业模式画布（9宫格）+ Phase 4 ATE 价值驱动
# 提示：用 pandas DataFrame 构建9宫格画布
#   列: '构件', 'MarketingAgent Pro', '传统SaaS对比'
#   9个构件: 客户细分/价值主张/渠道/客户关系/收入流/核心资源/核心活动/核心伙伴/成本结构
#   MarketingAgent Pro 的价值主张需整合 Phase 4 因果效果 (ATE=3.8pp转化率提升)
#   传统SaaS对比列展示AI适配的变化
# 要求：构建DataFrame并打印

# ===== 你的代码 =====
canvas_df = None  # 构建9宫格画布DataFrame

raise NotImplementedError


## 2. Phase 4 ATE -> ARPU 推导 + DCF 估值模型

**Capstone核心整合**：将Phase 4因果效果（ATE）转化为商业价值（ARPU），再通过DCF模型评估投资可行性。

推导链：ATE -> 月增转化 -> 月增收 -> ARPU -> DCF -> NPV

| 参数 | 值 | 来源 |
|------|-----|------|
| ATE | 0.038 (3.8pp) | Phase 4 因果推断（DoWhy） |
| 月触达 | 10,000 | 营销Agent系统基准 |
| AOV | $158 | 行业平均订单价值 |
| 价值捕获率 | 3.33% | outcome-based pricing (alpha) |
| 初始投资 | $2,000K | 开发团队 + GTM 投入 |
| 毛利率 | 65% | 含推理成本 30% + 数据 5% |
| 折现率 | 15% | VC 典型 SaaS 要求回报 |
| 评估窗口 | 5年 | J 曲线效应需 3-5 年 |

**numpy-financial 核心函数**：
- `npf.npv(rate, cashflows)` - 净现值
- `npf.irr(cashflows)` - 内部收益率


In [ ]:
# TODO 2：Phase 4 ATE -> ARPU 推导 + DCF 5年模型 + NPV
# 提示：1) Phase 4 因果效果参数:
#          ate = 0.038  (3.8pp转化率提升, 95%CI [0.022, 0.054])
#          monthly_reach = 10000  (每客户月均触达prospect数)
#          aov = 158  (平均订单价值$)
#          capture_rate = 0.0333  (价值捕获率, outcome-based pricing)
#       2) ARPU推导: arpu_monthly = reach * ATE * AOV * capture_rate
#          arpu_annual_k = arpu_monthly * 12 / 1000  ($K)
#       3) DCF参数:
#          initial_investment = 2000 ($K), discount_rate = 0.15
#          customers = [0, 30, 80, 160, 260, 380]
#          gross_margin = 0.65 (推理成本30% + 数据5%)
#          opex = [0, 800, 1200, 1800, 2500, 3200]
#       4) fcf = gross_profit - opex, fcf[0] = -initial_investment
#       5) npv = npf.npv(discount_rate, fcf)
# 要求：打印ARPU推导链 + DCF表 + NPV

# ===== 你的代码 =====

raise NotImplementedError


In [ ]:
# TODO 3：IRR + 回收期 + 盈利指数（PI）
# 提示：1) irr = npf.irr(fcf)
#       2) 回收期: 累计现金流首次转正的时点(手动计算)
#          cumulative += cf, 当 cumulative >= 0 时:
#          payback = (i-1) + (-prev_cum) / cf
#       3) PI = PV(未来现金流) / |初始投资|
#          pv_future = sum(fcf[i] / (1+dr)**i for i in range(1, len(fcf)))
# 要求：计算并打印IRR、回收期、PI，判断投资可行性

# ===== 你的代码 =====

raise NotImplementedError


## 3. 蒙特卡洛模拟（Monte Carlo Simulation）

DCF 给出 NPV 的点估计，但 AI SaaS 的关键参数高度不确定：
- **Phase 4 ATE**：因果效果估计有置信区间（95% CI: [2.2pp, 5.4pp]）
- **推理成本**：模型 API 价格快速变化（GPT-4 -> DeepSeek 成本降 90%+）
- **客户增长**：市场竞争 + 产品成熟度不确定
- **毛利率**：推理成本曲线决定长期毛利

蒙特卡洛方法：对不确定参数抽样（含ATE） -> 计算每次抽样的 NPV -> 得到估值分布

**scipy.stats / numpy 分布**：
- `np.random.normal(mu, sigma, n)` - 正态分布抽样
- `np.clip(arr, low, high)` - 截断分布范围
- `np.percentile(arr, q)` - 分位数


In [ ]:
# TODO 4：蒙特卡洛模拟（10000次）- 传播Phase 4 ATE不确定性
# 提示：1) 定义参数分布 (整合Phase 4 ATE置信区间):
#    - ATE ~ Normal(0.038, 0.008), clip to [0.010, 0.070]
#      (Phase 4 95% CI: [0.022, 0.054] -> sigma=0.008)
#    - Gross margin ~ Normal(0.65, 0.05), clip to [0.35, 0.85]
#    - Growth multiplier ~ Normal(1.0, 0.2), clip to [0.5, 1.5]
#    - OpEx multiplier ~ Normal(1.0, 0.15)
#       2) 每次抽样: 由ATE推导ARPU, 缩放客户数/收入/成本, 计算NPV
#       3) arpu_k = monthly_reach * ate_i * aov * capture_rate * 12 / 1000
#       4) 统计: 均值/中位数/标准差/P5/P95/P(NPV>0)
# 要求：打印统计量

# ===== 你的代码 =====

raise NotImplementedError


## 4. 敏感性分析（龙卷风图）

龙卷风图（Tornado Chart）展示各参数对 NPV 的影响排序：
- 对每个参数 +/-20% 变动，计算 NPV 变化范围
- 按影响大小降序排列，形成龙卷风形状
- 识别**高杠杆点**：小投入改变大局的关键参数

**2026前沿 - 推理成本对 AI 估值的影响**：
推理成本是 AI SaaS 估值的核心变量。DeepSeek 等开源模型将推理成本降低 90%+，
直接提升毛利率和估值。本Phase新增 **Phase 4 ATE** 作为敏感性参数，量化因果效果对NPV的影响。


In [ ]:
# TODO 5：敏感性分析 + 龙卷风图
# 提示：1) 定义 calc_npv(ate, inference_ratio, data_ratio, growth_mult, opex_mult, dr) 辅助函数
#          margin = 1 - inference_ratio - data_ratio
#          arpu_k = monthly_reach * ate * aov * capture_rate * 12 / 1000
#          基准: ate=0.038, inference_ratio=0.30, data_ratio=0.05
#       2) 对5个参数各+/-20%: ATE/Inference Cost/Growth/OpEx/Discount Rate
#       3) 按影响大小排序, 绘制龙卷风图
# 要求：打印敏感性表 + 绘制龙卷风图

# ===== 你的代码 =====

raise NotImplementedError


## 5. 天道推演 x 投资评估（2026前沿）

> 与项目 CLAUDE.md「天道推演系统」同构。

天道推演是一种元认知沙盘推演能力--以天神视角俯视局势，构建无限可能的沙盘，
模拟不同决策路径下的未来走向。应用于投资评估：

| 天道推演能力 | 投资评估对应 | 实现方式 |
|-------------|------------|---------|
| 局势感知 | 市场环境建模 | 场景定义（含Phase 4 ATE） |
| 因果链追踪 | 价值驱动因素分析 | 敏感性分析 |
| 沙盘模拟（3层） | 多路径推演 | Bull / Base / Bear |
| 概率评估 | 估值概率分布 | 蒙特卡洛模拟 |
| 最优路径推荐 | 投资决策 | NPV / IRR / PI |

**三路径推演**：Bull（乐观, ATE=CI上界5.4pp）/ Base（基准, ATE=3.8pp）/ Bear（悲观, ATE=CI下界2.2pp），每路径推演 3 层（immediate / near / far）。


In [ ]:
# TODO 6：天道推演多路径场景分析
# 提示：1) 定义3个场景: Bull/Base/Bear, 各含5个参数
#    Bull: ate=0.054(Phase4 CI上界), inference_ratio=0.23, growth_mult=1.3, opex_mult=0.9, dr=0.12
#    Base: ate=0.038, inference_ratio=0.30, growth_mult=1.0, opex_mult=1.0, dr=0.15
#    Bear: ate=0.022(Phase4 CI下界), inference_ratio=0.40, growth_mult=0.7, opex_mult=1.2, dr=0.20
#       2) 每场景计算NPV + 3层推演(immediate Y1-2 / near Y3-4 / far Y5)
#       3) 风险预警: 乐观-悲观跨度 + 关键风险 + 缓解策略
# 要求：打印三路径推演表 + 风险预警

# ===== 你的代码 =====

raise NotImplementedError


## 6. 反思与前沿

### 反思问题
1. MarketingAgent Pro 的 NPV 是多少？IRR 是否高于折现率？投资可行吗？
2. Phase 4 ATE 如何传导为 ARPU 和 NPV？推导链中哪个环节不确定性最大？
3. 蒙特卡洛模拟的 P(NPV>0) 是多少？5% 和 95% 分位差距说明了什么？
4. 敏感性分析中哪个参数对 NPV 影响最大？ATE 排第几？推理成本排第几？
5. 天道推演的三场景中，Bear case 的 NPV 是多少？风险预警是什么？

### 2026前沿：贝叶斯估值（Bayesian Valuation）
传统 DCF 给出点估计 NPV，蒙特卡洛给出频率派分布。**贝叶斯估值**用 PyMC 构建参数的
后验分布，结合先验信息和Phase 4观测数据，给出更稳健的估值后验分布。

### Phase 4-5 整合（Capstone核心）
| Phase | 能力 | Phase 5 整合角色 |
|-------|------|-----------------|
| Phase 4 | 因果实验设计与验证 | ATE -> ARPU -> NPV 推导链 |
| Phase 5 | 商业模式与价值评估 | 画布 + DCF + 蒙特卡洛 + 天道推演 |

### 技能4 Day 1-5 整合
| Day | 能力 | Phase 5 整合角色 |
|-----|------|-----------------|
| Day 1 | AI 商业模式类型学 | 画布的客户细分 + 价值主张 |
| Day 2 | AI 定价策略 | 画布的收入流（outcome-based） |
| Day 3 | Agent 经济学 | 画布的成本结构（推理成本） |
| Day 4 | 平台生态战略 | 画布的核心伙伴 + 渠道 |
| Day 5 | 商业模式画布 + 投资评估 | 画布框架 + NPV/IRR评估 |
